# F2 homework: a small data analysis, start to finish

In this notebook you analyse a dataset of exam results from three schools, using everything from F2:
NumPy, pandas and matplotlib. You will follow the workflow from the last lesson:
**question, load, inspect, clean, analyse, plot, conclude.**

**The question:** *How are study hours, sleep and school related to exam scores?*

The data is **synthetic**: generated by code with a fixed seed, so everyone gets exactly the same numbers. It has
been made messy on purpose, the way real data usually is.

Each task has a `TODO` cell and a check cell after it that prints `OK` when your code is right. Do not edit the check
cells. If a check fails, read its message: it tells you what is wrong.

**Where to run it**
- **Google Colab:** File > Upload notebook. NumPy, pandas and matplotlib are already installed.
- **Locally:** in your project folder from F0, run `uv add numpy pandas matplotlib`, then
  `uv run --with jupyter jupyter lab` and open this file.

## Setup

Run this cell first. It imports the three libraries with their usual nicknames.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("numpy", np.__version__, "| pandas", pd.__version__)

## The data

Run this cell to create two tables:

- `students_raw`: one row per student, with `student_id`, `school`, `study_hours` (per day), `sleep_hours`
  (per night) and `score` (exam score out of 100);
- `schools`: one row per school, with its `type` (state or private) and `town`.

You do not need to understand how `make_data` works, and **do not change it**: the checks rely on its exact output.

In [ ]:
def make_data(seed=2026):
    """Return (students, schools): a messy synthetic exam table and a small lookup table."""
    rng = np.random.default_rng(seed)
    n = 300
    school = rng.choice(["North", "South", "East"], size=n, p=[0.4, 0.35, 0.25])
    study = rng.uniform(0, 8, size=n).round(1)
    sleep = rng.normal(7, 1, size=n).round(1)
    effect = np.select([school == "South", school == "East"], [5, -3], 0)
    score = 30 + 6 * study + 3 * (sleep - 7) + effect + rng.normal(0, 8, size=n)
    students = pd.DataFrame({
        "student_id": np.arange(1001, 1001 + n),
        "school": school,
        "study_hours": study,
        "sleep_hours": sleep,
        "score": np.clip(score, 0, 100).round(),
    })
    # Real data is messy. These problems are deliberate.
    students.loc[rng.choice(n, size=9, replace=False), "study_hours"] = np.nan
    students.loc[[14, 88], "score"] = -1
    students.loc[201, "score"] = 999
    students.loc[[3, 50, 120], "school"] = ["north", " South", "EAST "]
    students = pd.concat([students, students.iloc[[7, 30, 31, 150]]], ignore_index=True)
    schools = pd.DataFrame({
        "school": ["North", "South", "East", "West"],
        "type": ["state", "private", "state", "private"],
        "town": ["Ashford", "Brayton", "Colby", "Dunmore"],
    })
    return students, schools


students_raw, schools = make_data()
students_raw.head()

## Task 1: inspect

Before changing anything, look at the data. Compute:

- `n_rows`: the number of rows in `students_raw`;
- `n_duplicates`: how many rows are exact duplicates of an earlier row (`students_raw.duplicated().sum()`);
- `missing`: a Series with the number of missing values per column;
- `school_spellings`: a **sorted list** of the distinct values in the `school` column;
- `n_bad_scores`: how many scores are impossible, meaning below 0 or above 100.

Also run `students_raw.describe()` and `students_raw.info()` and look at the output: what looks wrong?

In [ ]:
n_rows = None
n_duplicates = None
missing = None
school_spellings = None
n_bad_scores = None

# TODO: compute the five values above, then look at describe() and info()

In [ ]:
assert n_rows == 304, f"n_rows should be 304, got {n_rows}"
assert n_duplicates == 4, f"n_duplicates should be 4, got {n_duplicates}"
assert isinstance(missing, pd.Series) and missing["study_hours"] == 9 and missing.sum() == 9, "missing should be students_raw.isna().sum(): 9 missing study_hours and nothing else"
assert isinstance(school_spellings, list) and len(school_spellings) == 6, f"There are 6 different spellings of the 3 schools, got {school_spellings}"
assert n_bad_scores == 3, f"n_bad_scores should count scores below 0 or above 100: expected 3, got {n_bad_scores}"
print("OK")

## Task 2: clean

Write `clean(df)` that returns a cleaned **copy** of the students table. In this order:

1. drop exact duplicate rows;
2. drop rows with an impossible score (below 0 or above 100);
3. drop rows where `study_hours` is missing;
4. tidy the school names so that every spelling becomes `"North"`, `"South"` or `"East"`
   (`.str.strip().str.title()`).

`students_raw` must not change. Then the cell stores the result in `students`.

*Hint: add `.copy()` after the last row filter, before you change the `school` column.*

In [ ]:
def clean(df):
    """Return a cleaned copy of the students table."""
    # TODO
    return df


students = clean(students_raw)
print(len(students_raw), "rows before,", len(students), "after")

In [ ]:
assert len(students_raw) == 304, "clean changed students_raw: work on a copy"
assert len(students) == 288, f"Expected 288 rows after cleaning (304 minus 4 duplicates, 3 impossible scores and 9 missing study hours), got {len(students)}"
assert not students.duplicated().any(), "There are still duplicate rows"
assert students["score"].between(0, 100).all(), "There are still impossible scores"
assert students.isna().sum().sum() == 0, "There are still missing values"
assert set(students["school"]) == {"North", "South", "East"}, f"School names should be North, South and East only, got {sorted(set(students['school']))}"
print("OK")

## Task 3: new columns

Add two columns to `students`:

- `passed`: `True` when the score is at least 50;
- `study_band`: the study hours cut into the bands `"0-2"`, `"2-4"`, `"4-6"` and `"6-8"` with
  `pd.cut(..., bins=[0, 2, 4, 6, 8], labels=[...], include_lowest=True)`.
  `include_lowest=True` makes a student with exactly 0 hours fall into the first band.

In [ ]:
# TODO: add the passed and study_band columns to students

In [ ]:
assert "passed" in students.columns and "study_band" in students.columns, "Add both columns to students"
assert students["passed"].equals(students["score"] >= 50), "passed should be True exactly when score >= 50"
assert students["study_band"].isna().sum() == 0, "Every student should get a band: did you use include_lowest=True?"
assert (students.loc[students["study_hours"] <= 2, "study_band"] == "0-2").all(), "Students with 2 or fewer hours belong in 0-2"
assert (students.loc[students["study_hours"] > 6, "study_band"] == "6-8").all(), "Students with more than 6 hours belong in 6-8"
assert (students.loc[students["study_hours"].between(2.05, 4), "study_band"] == "2-4").all(), "Students with more than 2 and up to 4 hours belong in 2-4"
print("OK")

## Task 4: merge

Add each student's school `type` and `town` from the `schools` table. Every student must be kept, and no rows
may be added. Store the result in `data`.

Then print the row counts before and after, as you should after every merge.

In [ ]:
data = None  # TODO

In [ ]:
assert isinstance(data, pd.DataFrame), "data should be a DataFrame"
assert len(data) == len(students) == 288, f"The merge should keep exactly the 288 students, got {len(data)} rows"
assert {"type", "town"} <= set(data.columns), "data should have type and town columns from schools"
assert data["type"].isna().sum() == 0, "Every student should have a school type: check the school names were cleaned"
assert "West" not in set(data["school"]), "West has no students, so it should not appear: use a left join"
print("OK")

## Task 5: correlation with NumPy

The **correlation** $r$ between two variables measures how closely they follow a straight-line relationship, from
$-1$ to $1$. One way to compute it: standardise both variables (lesson 3), then average the products,

$$r = \frac{1}{n}\sum_{i=1}^{n} z_{x,i}\, z_{y,i}$$

where $z_{x,i}$ is the standardised value of $x$ for student $i$ and $n$ is the number of students.

Using **NumPy arrays and broadcasting, with no loops**:

1. `X = data[["study_hours", "sleep_hours"]].to_numpy()` (shape `(288, 2)`) and `y = data["score"].to_numpy()`;
2. `Z`: `X` with each column standardised (mean 0, standard deviation 1);
3. `zy`: `y` standardised;
4. `r = (Z * zy[:, None]).mean(axis=0)`: one correlation per column. Make sure you understand why `[:, None]` is needed.

Then store `r_study = r[0]` and `r_sleep = r[1]`.

In [ ]:
X = None
y = None
Z = None
zy = None
r = None
r_study = None
r_sleep = None

# TODO

In [ ]:
assert X.shape == (288, 2) and y.shape == (288,), f"X should have shape (288, 2) and y (288,), got {X.shape} and {y.shape}"
np.testing.assert_allclose(Z.mean(axis=0), [0, 0], atol=1e-9, err_msg="Each column of Z should have mean 0")
np.testing.assert_allclose(Z.std(axis=0), [1, 1], err_msg="Each column of Z should have standard deviation 1")
np.testing.assert_allclose(r_study, np.corrcoef(X[:, 0], y)[0, 1], err_msg="r_study does not match np.corrcoef")
np.testing.assert_allclose(r_sleep, np.corrcoef(X[:, 1], y)[0, 1], err_msg="r_sleep does not match np.corrcoef")
print("OK")

## Task 6: group and compare

Build three summary tables with `groupby` and named aggregation:

- `by_school`: one row per school, with columns `n` (number of students), `mean_score` and `pass_rate`
  (the mean of `passed`, which is the fraction who passed), sorted by `mean_score`, highest first;
- `by_band`: the same three columns for each `study_band`, in band order (pass `observed=True` to `groupby`);
- `by_type`: the same three columns for each school `type`.

Print all three, rounded to 2 decimal places, and look at the `n` column before believing any mean.

In [ ]:
by_school = None
by_band = None
by_type = None

# TODO

In [ ]:
for name, t in [("by_school", by_school), ("by_band", by_band), ("by_type", by_type)]:
    assert isinstance(t, pd.DataFrame), f"{name} should be a DataFrame"
    assert {"n", "mean_score", "pass_rate"} <= set(t.columns), f"{name} needs the columns n, mean_score and pass_rate"
    assert t["n"].sum() == 288, f"The n column of {name} should add up to 288"
    assert t["pass_rate"].between(0, 1).all(), f"pass_rate in {name} should be a fraction between 0 and 1"
assert list(by_school.index) == list(by_school.sort_values("mean_score", ascending=False).index), "Sort by_school by mean_score, highest first"
assert list(by_school.index)[0] == "South", "Check your by_school table: South should have the highest mean score"
assert [str(b) for b in by_band.index] == ["0-2", "2-4", "4-6", "6-8"], "by_band should have the four bands in order"
assert sorted(by_type.index) == ["private", "state"], "by_type should have one row per school type"
print("OK")

## Task 7: plot

Make **one figure with three plots side by side** (`plt.subplots(1, 3, figsize=(15, 4))`) and store it in `fig`:

1. a **histogram** of `score` with 20 bins;
2. a **scatter plot** of `score` against `study_hours`, with one colour per school `type` and a legend
   (loop over `data.groupby("type")`, as in the last lesson);
3. a **bar chart** of `pass_rate` per school from `by_school`.

Give every plot a title and axis labels, with units where there are any (hours, points). Then look at your plots: do they
agree with your tables?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# TODO: draw on axes[0], axes[1] and axes[2]

fig.tight_layout()

In [ ]:
assert len(fig.axes) == 3, f"The figure should have 3 axes, found {len(fig.axes)}"
a0, a1, a2 = fig.axes
assert len(a0.patches) == 20, f"The histogram should have 20 bins, found {len(a0.patches)}"
assert len(a1.collections) >= 2 and a1.get_legend() is not None, "The scatter plot needs one series per school type and a legend"
assert len(a2.patches) == 3, f"The bar chart should have one bar per school (3), found {len(a2.patches)}"
for i, ax in enumerate(fig.axes):
    assert ax.get_title() and ax.get_xlabel() and ax.get_ylabel(), f"Plot {i + 1} needs a title and both axis labels"
print("OK")

## Task 8: findings

Write **three findings** in the cell below, in the style of the last lesson. Each one must:

- quote at least one specific number from your tables (a mean, a rate or a correlation);
- say how many students it is based on;
- use careful language: "is associated with", not "causes".

Then add a short **Limitations** paragraph: how many rows did cleaning remove, and why might these results not carry
over to real schools? (Remember how the data was made.)

*Double-click to edit this cell.*

1. **Finding 1:** ...
2. **Finding 2:** ...
3. **Finding 3:** ...

**Limitations:** ...

## Stretch (optional)

1. Instead of dropping the 9 students with missing `study_hours`, fill them with the median study time. Rebuild `by_band`.
   Do any of your findings change? Which choice would you report, and why?
2. School type and school are tangled together here: South is the only private school. Can this data tell you whether
   *private schools* do better, or only whether *South* does? Write two sentences.
3. Try the histogram with 5 and with 100 bins. Which bin count would you choose to show someone, and why?

In [ ]:
# Your code here

## Before you finish

Go through the rubric on the homework page and tick each item you have done. Then run **Kernel > Restart and run all**
(on Colab: **Runtime > Restart session and run all**) to make sure every check cell prints `OK` from a clean start.